# Lab 02 · Embeddings, 3D PCA and modality

Module C • **Core offline toy vectors** + **optional real MiniLM model**. Toy vectors explain geometry; they are not learned embeddings.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**C1 · NAT · after cell 02 · C1** — How many records are in the supplied toy vector corpus?  
NAT: `_________`

**C2 · NAT · after cell 02 · C1** — How many dimensions does each toy input vector have?  
NAT: `_________`

**C3 · NAT · after cell 02 · C3** — How many components does PCA create in the notebook?  
NAT: `_________`

**C4 · NAT · after cell 02 · C3** — How many rows will the PCA coordinate matrix contain?  
NAT: `_________`

**C5 · MCQ · after cell 02 · C3** — What does PCA choose?
  - **A.** Three random original features
  - **B.** Axes of greatest variance
  - **C.** Three nearest neighbors
  - **D.** New words generated by an LLM

**C6 · MCQ · after cell 02 · C5** — Are distances always identical before and after 6D→3D PCA?
  - **A.** Yes, PCA preserves every distance
  - **B.** No, discarded dimensions can change distances
  - **C.** Only if the labels are text
  - **D.** Yes, when plotting in 3D

**C7 · MCQ · after cell 02 · C1** — What do the supplied image/video toy vectors demonstrate?
  - **A.** Proven real multimodal embeddings
  - **B.** A visually useful *illustration* of a shared space
  - **C.** An image encoder trained on ImageNet
  - **D.** Gemini embedding API output

**C8 · MCQ · after cell 02 · C6** — The optional MiniLM text model normally outputs how many dimensions?
  - **A.** 3
  - **B.** 6
  - **C.** 384
  - **D.** 3072

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=C) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [scikit-learn PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) · [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) · [Gemini embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## C1 · The tiny shared-space *illustration*
We deliberately designed the numbers below so cat-related media live near each other. This does **not** prove real text/image/video models share a space. A real cross-modal space requires a model trained for that task.

In [ ]:
import json
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
items=json.loads((ROOT/'data/toy_vectors.json').read_text())
labels=[item['label'] for item in items]
X=np.array([item['vector'] for item in items],dtype=float)
print('input_shape =',X.shape)
print('labels =',labels)
print('first_vector_dimensions =',len(X[0]))

## C2 · Make a cosine-similarity matrix

In [ ]:
similarities=cosine_similarity(X)
df=pd.DataFrame(np.round(similarities,3),index=labels,columns=labels)
display(df)
cat_ix=labels.index('Cat')
ranking=sorted([(labels[i],float(similarities[cat_ix,i])) for i in range(len(labels)) if i!=cat_ix], key=lambda a:-a[1])
print('top 4 neighbors of Cat (toy vectors):',[(x,round(y,3)) for x,y in ranking[:4]])

## C3 · PCA: 6 dimensions → 3 dimensions
PCA finds orthogonal directions of highest variance in **this dataset**. It is a visualization technique, *not* the original embedding model and not an exact similarity-preserving transform.

In [ ]:
pca=PCA(n_components=3)
coords=pca.fit_transform(X)
print('pca_shape =',coords.shape)
print('variance_explained_by_3 =',round(float(pca.explained_variance_ratio_.sum()),4))
print(pd.DataFrame(np.round(coords,3),index=labels,columns=['PC1','PC2','PC3']))

## C4 · Interactive 3D projection: rotate, pan and hover

In [ ]:
import plotly.express as px
fig=px.scatter_3d(x=coords[:,0],y=coords[:,1],z=coords[:,2],text=labels,color=[i['kind'] for i in items],
                  labels={'x':'PC1','y':'PC2','z':'PC3','color':'Input type'},title='PCA of illustrative vectors — rotate with mouse')
fig.update_traces(marker=dict(size=7),textposition='top center')
fig.show()

## C5 · Compare distance before and after projection
Choose **Cat** and **Cat image**, then compare squared L2 in the original 6D space versus PCA 3D. Because PCA discards components, distances may differ.

In [ ]:
i,j=labels.index('Cat'),labels.index('Cat image')
d6=float(np.sum((X[i]-X[j])**2))
d3=float(np.sum((coords[i]-coords[j])**2))
print('squared_l2_6d =',round(d6,7))
print('squared_l2_pca3d =',round(d3,7))
print('distances_exactly_identical =',np.isclose(d6,d3,atol=1e-12))

## C6 · Compare a real embedding model (OPTIONAL: model download)
This is a **learned** text embedding model. It will download weights the first time and is not essential to pass the offline quiz. Do not call these vectors multimodal.

In [ ]:
# OPTIONAL CELL — un-comment the next line in a new cell if needed:
# %pip install -q sentence-transformers
RUN_MINILM = False  # ✏️ Set True if you want to download and run the real model.
if not RUN_MINILM:
    print('OPTIONAL SKIPPED: set RUN_MINILM=True to use learned MiniLM embeddings.')
else:
    from sentence_transformers import SentenceTransformer
    model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    texts=['The cat is on the table','A kitten rests on the desk','A car drives along the road','My phone has a camera']
    real_embeddings=model.encode(texts,normalize_embeddings=True)
    print('MiniLM real embedding shape =',real_embeddings.shape)
    print('Real cosine matrix:')
    display(pd.DataFrame(np.round(cosine_similarity(real_embeddings),3),index=texts,columns=texts))

### Finish Module C
Answer **C1–C8** using the deterministic core cells. Compare why the optional real model sometimes gives unexpected rankings.

## Hints · Use these only after attempting the tasks

- **C1:** Print `len(items)` and inspect unique IDs.
- **C2:** Inspect `X.shape[1]`.
- **C3:** Read `PCA(n_components=...)` in cell C3.
- **C4:** PCA produces one 3D row for every original item.
- **C5:** PCA uses variance directions, not nearest neighbors.
- **C6:** Compare d6 and d3 from cell C5, not the 3D picture alone.
- **C7:** Look at how the `toy_vectors.json` values were constructed.
- **C8:** Check the official all-MiniLM-L6-v2 model card.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**C1: 10**  
Ten records; each has a distinct id.

**C2: 6**  
Each vector has six authored coordinates.

**C3: 3**  
n_components=3.

**C4: 10**  
PCA projects each of ten input vectors.

**C5: B — Axes of greatest variance**  
PCA finds orthogonal directions accounting for variance in the dataset.

**C6: B — No, discarded dimensions can change distances**  
3D is a lossy projection unless data lives entirely in 3 dimensions.

**C7: B — A visually useful *illustration* of a shared space**  
They were manually authored. Real cross-modal alignment requires a trained multimodal model.

**C8: C — 384**  
all-MiniLM-L6-v2 uses 384-dimensional sentence embeddings.

### Extension challenge — Interview explanation

**Challenge:** A cat image looks close to cat text in a 3D projection. What additional evidence would show true semantic cross-modal alignment?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
